In [ ]:
# Import necessary libraries.
%matplotlib inline

from pycbc.waveform import get_td_waveform
import pycbc.waveform
import pylab
pylab.rcParams['figure.dpi'] = 100
import numpy as np

## Q transform vs. FFT comparision

Generate a sequence of random numbers (white noise) and analyse its frequency contributions.

Perform FFT analysis and watch the equal spacing of the components on the linear scale of frequencies.

In [ ]:
N = 5120                     # Number of samples
Fs = 1000                   # Sampling frequency

time = np.linspace(0, N/Fs, N)  # Generate 1024 time labels between 0 and 1 second.

my_rng = np.random.default_rng()

our_signal = my_rng.standard_normal(N)       # Generate 1024 samples of the sinusoidal signal.

pylab.plot(time, our_signal)       # Plot the signal (Note: the frequency 16Hz)

In [ ]:
from scipy.fftpack import fft, fftfreq, fftshift

fft = fft(our_signal,n=N)

fft = fft[1:int(N/2+1)]  # Take just positive frequencies (see jupyter 4-Discrete Fourier Transform)

freq = np.fft.fftfreq(N, d=1./Fs)

psd = (1/(Fs*N)) * np.power(np.abs(fft), 2)
psd2 = 2*psd

pylab.xlabel('frequency [Hz]')
pylab.ylabel('PSD [V**2/Hz]')
pylab.semilogy(freq[0:int(N/2)], psd2)

# Perform Q transform analysis.

Plot the frequency scale <i>f</i>. In the contrary to the FFT scale it is not linear.

The plot you obtain is the typical image of the white noise.

In [ ]:
# It is easier to call the qtransform() function on the object of type TimeSeries.
our_signal_ts = pycbc.types.timeseries.TimeSeries(our_signal, delta_t=1./Fs)

t, f, p = our_signal_ts.qtransform(.001,
                         logfsteps=100,
                         qrange=(12,12),
                         frange=(5, 400))

pylab.figure(figsize=[15, 3])
pylab.title('GW Data')
pylab.pcolormesh(t, f, np.log(p**0.5), vmin=0, vmax=2, shading='auto')
pylab.yscale('log')
pylab.xlabel('Time (s)')
pylab.ylabel('Frequency (Hz)')
pylab.show()

In [ ]:
print(p.shape)

pylab.plot(f)

In [ ]:
# Slice of the upper plot in t and compare it to the FFT plot in logaritmic frequency scale.
# Frequency grid is logaritmic in this case

pylab.plot(f, p[:,0])
#pylab.plot(f, p[:,])
pylab.xscale('log')

In [ ]:
pylab.loglog(freq[0:int(N/2)], psd2)

# Analyze a waveform generated by the SEOBNRv4HM model via the Q-transform

Simulate GW150914 (https://en.wikipedia.org/wiki/First_observation_of_gravitational_waves) with the parameters:
* $\text{distance} = 440 {Mpc}$
* $M_1 = 35 M_\odot$
* $M_2 = 30 M_\odot$

In [ ]:
hp, hc = get_td_waveform(approximant="SEOBNRv4HM",                         
                         mass1=35,
                         mass2=30,
                         spin1x=0,  # For v4HM Must be zero lalsimulation/lib/LALSimInspiral.c
                         spin1y=0,  # Must be zero
                         spin1z=0,
                         spin2x=0,  # Must be zero
                         spin2y=0,  # Must be zero
                         spin2z=0,
                         distance=440,
                         coa_phase=1.2,
                         inclination=0,
                         long_asc_nodes=0,
                         eccentricity=0,
                         mean_per_ano=0,
                         delta_t=1.0/4096,
                         f_lower=20)

pylab.figure(figsize=[15, 3])
pylab.plot(hp.sample_times, hp, label="Data")
pylab.legend()
pylab.show()

# Prepare the signal for Q transform.

1. Inclease length of the timeseries
2. Time-shift the signal (subsequent filtering will case a phaseshift)
3. Filter the signal to remove high and low frequencies

In [ ]:
hp.resize(2.*len(hp))

In [ ]:
template = hp.cyclic_time_shift(hp.start_time-2.2)

wfm_filtered = template.highpass_fir(20, 512).lowpass_fir(600, 512)

pylab.plot(wfm_filtered.sample_times, wfm_filtered, label="Data")

# Compute and display the Q transform

In [ ]:
t, f, p = wfm_filtered.qtransform(.001,
                         logfsteps=100,
                         qrange=(8, 8),
                         frange=(20, 1024))
pylab.figure(figsize=[15, 3])
pylab.title('GW Data')
pylab.pcolormesh(t, f, np.log(p**0.5), vmin=3, vmax=12, shading='auto')
pylab.yscale('log')
pylab.xlabel('Time (s)')
pylab.ylabel('Frequency (Hz)')
pylab.xlim(3.2, 3.6)
pylab.show()